In [0]:
%spark.pyspark 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

hadoop_conf = spark._jsc.hadoopConfiguration()


hadoop_conf.set("fs.s3a.access.key", "---")
hadoop_conf.set("fs.s3a.secret.key", "---")

hadoop_conf.set("fs.s3a.endpoint", "storage.yandexcloud.net")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")


hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "true")



users = spark.createDataFrame(
    [ 
        ("u1", "Berlin"),
        ("u2", "Berlin"),
        ("u3", "Munich"),
        ("u4", "Hamburg"), 
    ],
    ["user_id", "city"]
)

orders = spark.createDataFrame(
    [ 
        ("o1", "u1", "p1", 2, 10.0),
        ("o2", "u1", "p2", 1, 30.0),
        ("o3", "u2", "p1", 1, 10.0),
        ("o4", "u2", "p3", 5, 7.0),
        ("o5", "u3", "p2", 3, 30.0),
        ("o6", "u3", "p3", 1, 7.0),
        ("o7", "u4", "p1", 10, 10.0), ],
        ["order_id", "user_id", "product_id", "qty", "price"]
)

products = spark.createDataFrame(
    [ 
        ("p1", "Ring VOLA"),
        ("p2", "Ring POROG"),
        ("p3", "Ring TISHINA"), 
    ],
    ["product_id", "product_name"] 
)

users.show() 
orders.show() 
products.show()

In [1]:
%spark.pyspark

orders = orders.withColumn("revenue", F.col("qty") * F.col("price"))
df = (
    orders
    .join(users, on="user_id", how="inner")
    .join(products, on="product_id", how="inner")
)
agg_df = (
    df.groupBy("city", "product_id", "product_name")
      .agg(
          F.countDistinct("order_id").alias("orders_cnt"),
          F.sum("qty").alias("qty_sum"),
          F.sum("revenue").alias("revenue_sum")
      )
)
window_spec = Window.partitionBy("city").orderBy(F.desc("revenue_sum"))

result_df = (
    agg_df
    .withColumn("rank", F.row_number().over(window_spec))
    .filter(F.col("rank") <= 2)
    .drop("rank")
)

In [2]:
%spark.pyspark

hdfs_path = "/tmp/sandbox_zeppelin/mart_city_top_products/"
s3_path = "s3a://s3s-dz/tmp/sandbox_zeppelin/mart_city_top_products"

result_df.write.mode("overwrite").parquet(hdfs_path)
result_df.write.mode("overwrite").parquet(s3_path)


loaded_hadoop_df = spark.read.parquet(hdfs_path)
loaded_hadoop_df.orderBy("city", F.desc("revenue_sum")).show()

df_s3 = spark.read.parquet(s3_path)
df_s3.show()

In [3]:
%spark.pyspark
